## Importing libraries 

In [16]:
import sounddevice as sd
import numpy as np
import time
import threading
import queue
import keyboard
from IPython.display import display, HTML
from sklearn.ensemble import RandomForestClassifier
import pickle

## Building the morse code dictionary 

In [17]:
morse_dict = {
    # Letters
    ".-": "A", "-...": "B", "-.-.": "C", "-..": "D", ".": "E",
    "..-.": "F", "--.": "G", "....": "H", "..": "I", ".---": "J",
    "-.-": "K", ".-..": "L", "--": "M", "-.": "N", "---": "O",
    ".--.": "P", "--.-": "Q", ".-.": "R", "...": "S", "-": "T",
    "..-": "U", "...-": "V", ".--": "W", "-..-": "X", "-.--": "Y",
    "--..": "Z",

    # Numbers
    "-----": "0", ".----": "1", "..---": "2", "...--": "3",
    "....-": "4", ".....": "5", "-....": "6", "--...": "7",
    "---..": "8", "----.": "9",

    # Common punctuation
    ".-.-.-": ".", "--..--": ",", "..--..": "?",
    ".----.": "'", "-.-.--": "!", "-..-.": "/",
    "-.--.": "(", "-.--.-": ")", ".-...": "&",
    "---...": ":", "-.-.-.": ";", "-...-": "=",
    ".-.-.": "+", "-....-": "-", "..--.-": "_",
    ".-..-.": "\"", "...-..-": "$", ".--.-.": "@"
}

## Configurable Parameters

In [23]:
# Timing & detection thresholds (tune these to your environment)
GAIN            = 100      # Software amplification of mic input
SPIKE_FACTOR    = 3.0     # A tap must be this many times LOUDER than the ambient noise floor
NOISE_SMOOTH    = 0.997   # How fast the noise floor adapts (0.997 = very stable)
DOT_DURATION    = 0.15    # Max seconds for a dot (longer = dash)
LETTER_GAP      = 0.40    # Silence seconds to finalise a letter
WORD_GAP        = 1.00    # Silence seconds to insert a space
SAMPLE_RATE     = 44100   # Audio sample rate in Hz

## Helper Functions

**Pipeline:**  
Audio Input → `detect_sound()` → `measure_duration()` → `classify_signal()` → `group_symbols()` → `decode_morse()` → Text Output

In [19]:
def measure_duration(start_time):
    """Return elapsed seconds since start_time."""
    return time.time() - start_time


def classify_signal(duration, dot_limit=DOT_DURATION):
    """Return '.' for a short tap, '-' for a long tap."""
    return "." if duration < dot_limit else "-"


def decode_morse(symbol_buffer, dictionary=None):
    """Look up a dot/dash sequence in the Morse dictionary.
    Returns the decoded character, or '?' if unknown."""
    if dictionary is None:
        dictionary = morse_dict
    return dictionary.get(symbol_buffer, "?")


def extract_features(block):
    """Extract audio features from a single frame for tap vs noise classification.
    Returns a 1D array of 6 features."""
    block = block.flatten()
    rms = np.sqrt(np.mean(block ** 2)) + 1e-10
    peak = np.max(np.abs(block))
    crest_factor = peak / rms                          # high for sharp taps
    zcr = np.sum(np.abs(np.diff(np.sign(block)))) / (2 * len(block))  # zero-crossing rate
    # Spectral centroid
    fft_mag = np.abs(np.fft.rfft(block))
    freqs = np.fft.rfftfreq(len(block), d=1.0 / SAMPLE_RATE)
    spectral_centroid = np.sum(freqs * fft_mag) / (np.sum(fft_mag) + 1e-10)
    # Attack: ratio of first-quarter energy to total energy
    q = len(block) // 4
    attack = np.sqrt(np.mean(block[:q] ** 2)) / rms if q > 0 else 1.0

    return np.array([rms, peak, crest_factor, zcr, spectral_centroid, attack])

## Calibration — Train the Tap Detector

Run the cell below. It will record two short sessions:
1. **5 seconds of silence / background noise** — just sit normally  
2. **5 seconds of tapping** — tap the table repeatedly  

A Random Forest classifier will learn the difference and filter out non-tap sounds during decoding.

In [20]:
# # ── Step 1: Record background noise ──────────────────────────
# print("🔇  Recording 5 seconds of BACKGROUND NOISE — do NOT tap...\n")
# bg_rec = sd.rec(int(5 * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
# sd.wait()
# bg_rec = bg_rec * GAIN
# print("   ✅ Background recorded.\n")

# # ── Step 2: Record taps ─────────────────────────────────────
# print("🔊  Recording 5 seconds of TAPS — tap the table repeatedly NOW!\n")
# tap_rec = sd.rec(int(5 * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
# sd.wait()
# tap_rec = tap_rec * GAIN
# print("   ✅ Taps recorded.\n")

# # ── Step 3: Extract features ────────────────────────────────
# blocksize = 1024
# X, y = [], []

# # Background samples → label 0
# for i in range(0, len(bg_rec) - blocksize, blocksize):
#     feats = extract_features(bg_rec[i:i+blocksize])
#     X.append(feats)
#     y.append(0)

# # Tap samples → label 1 (only frames loud enough to possibly be a tap)
# bg_rms = np.mean([np.sqrt(np.mean(bg_rec[i:i+blocksize]**2))
#                    for i in range(0, len(bg_rec)-blocksize, blocksize)])

# for i in range(0, len(tap_rec) - blocksize, blocksize):
#     block = tap_rec[i:i+blocksize]
#     block_rms = np.sqrt(np.mean(block**2))
#     feats = extract_features(block)
#     # Only label loud frames as taps; quiet frames in tap recording are still noise
#     if block_rms > bg_rms * 2:
#         X.append(feats)
#         y.append(1)
#     else:
#         X.append(feats)
#         y.append(0)

# X = np.array(X)
# y = np.array(y)

# print(f"   Training samples: {len(y)} ({np.sum(y==0)} noise, {np.sum(y==1)} tap)")

# # ── Step 4: Train classifier ────────────────────────────────
# tap_model = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42)
# tap_model.fit(X, y)

# train_acc = tap_model.score(X, y)
# print(f"   Training accuracy: {train_acc:.1%}")

# # ── Step 5: Save model to disk ──────────────────────────────
# MODEL_PATH = "tap_model.pkl"
# with open(MODEL_PATH, "wb") as f:
#     pickle.dump(tap_model, f)
# print(f"   💾 Model saved to {MODEL_PATH}")
# print(f"\n✅  Tap detector model ready!")

## Real-Time Morse Code Listener (ML-Filtered)

The listener now uses the trained model to verify each detected sound is actually a tap before recording it as a dot or dash.

In [24]:
# ── Load saved model ──────────────────────────────────────────
MODEL_PATH = "tap_model.pkl"
with open(MODEL_PATH, "rb") as f:
    tap_model = pickle.load(f)
print(f"✅ Loaded tap model from {MODEL_PATH}\n")

# ── Shared state ──────────────────────────────────────────────
is_sound        = False
start_time      = 0.0
last_sound_time = 0.0
symbol_buffer   = ""
decoded_text    = ""
gap_checked     = False
current_volume  = 0.1
noise_floor     = 0.01
current_thresh  = 0.0
ml_label        = "—"
tap_buffer      = []           # accumulate audio frames while tap is active

lock = threading.Lock()
output_q = queue.Queue()

# ── Audio callback (runs per audio frame) ─────────────────────
def callback(indata, frames, time_info, status):
    global is_sound, start_time, last_sound_time, symbol_buffer, gap_checked
    global current_volume, noise_floor, current_thresh, ml_label, tap_buffer

    amplified = indata * GAIN
    volume = np.linalg.norm(amplified) / len(amplified) ** 0.5
    current_volume = volume

    # Adaptive noise floor (only update during silence)
    if not is_sound:
        noise_floor = NOISE_SMOOTH * noise_floor + (1 - NOISE_SMOOTH) * volume

    threshold = noise_floor * SPIKE_FACTOR
    current_thresh = threshold

    with lock:
        if volume > threshold:
            # ── Spike detected (threshold only — fast) ────────
            if not is_sound:
                is_sound = True
                start_time = time.time()
                gap_checked = False
                tap_buffer = [amplified.copy()]
            else:
                tap_buffer.append(amplified.copy())
            ml_label = "detecting…"
        else:
            if is_sound:
                # ── Tap just ended — validate with ML ─────────
                duration = measure_duration(start_time)
                is_sound = False
                last_sound_time = time.time()

                # Find the loudest 1024-sample chunk (matches training block size)
                full_tap = np.concatenate(tap_buffer).flatten()
                chunk_size = 1024
                if len(full_tap) >= chunk_size:
                    best_rms, best_chunk = 0, full_tap[:chunk_size]
                    for ci in range(0, len(full_tap) - chunk_size + 1, chunk_size // 2):
                        c = full_tap[ci:ci+chunk_size]
                        r = np.sqrt(np.mean(c**2))
                        if r > best_rms:
                            best_rms, best_chunk = r, c
                    feats = extract_features(best_chunk).reshape(1, -1)
                else:
                    feats = extract_features(full_tap).reshape(1, -1)
                tap_prob = tap_model.predict_proba(feats)[0][1]

                if tap_prob > 0.1:
                    symbol = classify_signal(duration)
                    symbol_buffer += symbol
                    output_q.put(("symbol", symbol))
                    ml_label = f"TAP ({tap_prob:.0%})"
                else:
                    ml_label = f"rejected ({tap_prob:.0%})"
                tap_buffer = []
            else:
                ml_label = "quiet"

# ── Gap-monitoring thread ─────────────────────────────────────
def group_symbols():
    global symbol_buffer, decoded_text, gap_checked

    while not stop_event.is_set():
        time.sleep(0.05)
        with lock:
            if is_sound or last_sound_time == 0.0:
                continue
            silence = time.time() - last_sound_time

            if silence >= WORD_GAP and symbol_buffer:
                letter = decode_morse(symbol_buffer)
                decoded_text += letter + " "
                output_q.put(("letter", letter))
                output_q.put(("space", " "))
                symbol_buffer = ""
                gap_checked = True
            elif silence >= LETTER_GAP and symbol_buffer and not gap_checked:
                letter = decode_morse(symbol_buffer)
                decoded_text += letter
                output_q.put(("letter", letter))
                symbol_buffer = ""
                gap_checked = True

# ── Build live HTML with volume meter ─────────────────────────
def render_html(raw_log, text, vol, threshold, floor, label):
    bar_max = 50
    cap = max(threshold * 3, 0.01)
    scale = min(vol / cap, 1.0)
    filled = int(scale * bar_max)

    is_active = vol > threshold and label.startswith("TAP")
    bar_color = "#4caf50" if is_active else "#666"
    bar = (
        f"<span style='color:{bar_color};'>{'█' * filled}</span>"
        f"<span style='color:#333;'>{'░' * (bar_max - filled)}</span>"
    )

    if is_active or label.startswith("TAP"):
        status = f"🟢 {label}"
    elif label.startswith("rejected"):
        status = f"🟡 {label}"
    elif label == "detecting…":
        status = "🔵 detecting…"
    else:
        status = "⚫ quiet"

    return HTML(
        f"<div style='font-family:monospace;font-size:14px;line-height:1.6;'>"
        f"<b>🎤 Mic:</b> |{bar}| {status} "
        f"<span style='color:#aaa;'>vol={vol:.4f}  thresh={threshold:.4f}  floor={floor:.4f}</span><br>"
        f"<b>Signals:</b> {raw_log if raw_log else '<i style=\"color:#999;\">no taps yet</i>'}<br><br>"
        f"<b style='font-size:18px;'>Decoded:</b> "
        f"<span style='font-size:22px; color:#2e7d32;'>{text}</span>"
        f"</div>"
    )

# ── Main loop ─────────────────────────────────────────────────
stop_event = threading.Event()
gap_thread = threading.Thread(target=group_symbols, daemon=True)

raw_log = ""
handle = display(render_html("", "waiting for taps...", 0.0, 0.01, 0.01, "—"), display_id=True)

print("Press ESC to stop.\n")

try:
    with sd.InputStream(callback=callback, channels=1, samplerate=SAMPLE_RATE):
        gap_thread.start()
        while True:
            if keyboard.is_pressed('esc'):
                break

            try:
                while True:
                    kind, value = output_q.get_nowait()
                    if kind == "symbol":
                        raw_log += value
                    elif kind == "letter":
                        raw_log += f" <b>[{value}]</b> "
                    elif kind == "space":
                        raw_log += "&nbsp;&nbsp;"
            except queue.Empty:
                pass

            handle.update(render_html(
                raw_log,
                decoded_text if decoded_text else "waiting for taps...",
                current_volume,
                current_thresh,
                noise_floor,
                ml_label
            ))
            time.sleep(0.1)
finally:
    stop_event.set()
    gap_thread.join(timeout=1)
    if symbol_buffer:
        letter = decode_morse(symbol_buffer)
        decoded_text += letter
        raw_log += f" <b>[{letter}]</b>"
    handle.update(render_html(
        raw_log,
        decoded_text if decoded_text else "(no taps detected)",
        0.0, current_thresh, noise_floor, "—"
    ))

✅ Loaded tap model from tap_model.pkl



Press ESC to stop.



## Mic Diagnostic

Run the cell below and **tap the table a few times** during the 5-second recording.  
It will show the volume levels your mic actually picks up so we can set the right threshold.

In [22]:
# # Record 5 seconds and show volume stats
# print(f"🎤  Recording for 5 seconds (GAIN = {GAIN}x) — TAP THE TABLE NOW!\n")
# duration_sec = 5
# recording = sd.rec(int(duration_sec * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
# sd.wait()

# # Apply software gain
# recording = recording * GAIN

# # Compute RMS per frame
# blocksize = 1024
# rms_values = []
# for i in range(0, len(recording) - blocksize, blocksize):
#     block = recording[i:i+blocksize]
#     rms = np.linalg.norm(block) / len(block) ** 0.5
#     rms_values.append(rms)

# rms_values = np.array(rms_values)
# mean_vol = rms_values.mean()
# max_vol = rms_values.max()
# adaptive_thresh = mean_vol * SPIKE_FACTOR

# print(f"  Min  volume: {rms_values.min():.6f}")
# print(f"  Max  volume: {max_vol:.6f}")
# print(f"  Mean volume (≈ noise floor): {mean_vol:.6f}")
# print(f"  Adaptive threshold (mean × {SPIKE_FACTOR}): {adaptive_thresh:.6f}")
# print(f"  Max / Mean ratio: {max_vol / mean_vol:.1f}x")
# print()

# if max_vol > adaptive_thresh:
#     print(f"  ✅ Taps ARE detectable — peaks are {max_vol/mean_vol:.1f}× above noise (need {SPIKE_FACTOR}×)")
# else:
#     print(f"  ⚠️  Taps NOT strong enough — peaks are only {max_vol/mean_vol:.1f}× above noise (need {SPIKE_FACTOR}×)")
#     print(f"      Try: increase GAIN or decrease SPIKE_FACTOR")
# print()

# # Visual timeline
# print("  Volume timeline (each char = ~50ms):")
# print("  ", end="")
# for v in rms_values[::2]:
#     if v > adaptive_thresh:
#         print("█", end="")
#     elif v > adaptive_thresh * 0.5:
#         print("▄", end="")
#     else:
#         print("░", end="")
# print()